# ViT-Adapter — Video Semantic Segmentation (Cityscapes)

Runs the official **[ViT-Adapter](https://github.com/czczup/ViT-Adapter)** model
(`Mask2Former + BEiT-Adapter-L`, trained on **Cityscapes**, 84.9 mIoU) on a video,
frame by frame, and returns a colour-segmented `.mp4`. Built for **dashcam / driving
footage** (road, car, person, sidewalk, traffic sign, …).

> ViT-Adapter is an *image* model — "VSS" here means per-frame segmentation
> stitched back into a video.

---
## ✅ Guidelines — read this first
1. **Set a GPU runtime:** `Runtime ▸ Change runtime type ▸ Hardware accelerator = GPU (T4)`.
2. **Run cells strictly top-to-bottom.**
3. **Cell 1 restarts the runtime** (that's normal). After it restarts, continue
   from **Cell 2** — do *not* re-run Cell 1.
4. First run takes **~12–18 min** (conda env + a 2.1 GB checkpoint). Later runs in
   the same session are instant from Cell 6 onward.
5. **Cell 6** is where you upload your video (`sample_dashcam.mp4`).
6. If anything errors, see **Troubleshooting** at the bottom — each known issue is
   already worked around in these cells.
---

### 0. Confirm a GPU is attached

In [ ]:
import torch
print('GPU available:', torch.cuda.is_available())
!nvidia-smi -L

### 1. Install conda — this **restarts the runtime** (expected)
After the restart, continue from Cell 2. Do **not** re-run this cell.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

### 2. Create Python 3.9 env + install the ViT-Adapter (OpenMMLab 1.x) stack
~8–10 min. `cudatoolkit-dev` brings the `nvcc` 11.3 compiler used in Cell 3.

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda create -y -n vit python=3.9
conda activate vit

# Torch 1.10 + CUDA 11.3 (matches the mmcv-full 1.4.2 prebuilt wheel)
pip install torch==1.10.0+cu113 torchvision==0.11.0+cu113 \
    --extra-index-url https://download.pytorch.org/whl/cu113

# OpenMMLab 1.x stack required by ViT-Adapter (mmcv-full uses a prebuilt wheel)
pip install mmcv-full==1.4.2 \
    -f https://download.openmmlab.com/mmcv/dist/cu113/torch1.10.0/index.html
pip install mmsegmentation==0.20.2 mmdet==2.22.0 timm==0.4.12 \
    yapf==0.40.1 "numpy==1.23.5" opencv-python-headless ftfy regex

# nvcc 11.3 (in the conda env) for compiling the custom CUDA op in Cell 3
conda install -y -c conda-forge cudatoolkit-dev=11.3

echo '--- sanity ---'
python -c "import torch, mmcv, mmseg, mmdet; print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| mmcv', mmcv.__version__, '| mmseg', mmseg.__version__)"


### 3. Clone ViT-Adapter & compile the deformable-attention CUDA op
We force **gcc-10** here because `nvcc` 11.3 refuses gcc ≥ 11 (Colab's default) —
this is the single most common build error, handled for you.

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda activate vit
PY=$CONDA_PREFIX/bin/python          # full path -> never shadowed by system python

# nvcc 11.3 needs gcc <= 10; make gcc-10/g++-10 the default for the build.
apt-get install -y -qq gcc-10 g++-10 > /dev/null
ln -sf /usr/bin/gcc-10 /usr/local/bin/gcc
ln -sf /usr/bin/g++-10 /usr/local/bin/g++
export CC=/usr/bin/gcc-10 CXX=/usr/bin/g++-10
export CUDA_HOME=$CONDA_PREFIX

cd /content
[ -d ViT-Adapter ] || git clone https://github.com/czczup/ViT-Adapter.git
cd ViT-Adapter/segmentation
ln -sf ../detection/ops ./ops
cd ops
$PY setup.py build install
$PY -c "import MultiScaleDeformableAttention; print('CUDA op compiled OK')"

In [ ]:
!/usr/local/envs/vit/bin/python -c "import torch, MultiScaleDeformableAttention; print('CUDA op OK')"

### 4. Download the Cityscapes checkpoint (~2.1 GB, a few minutes)

In [ ]:
%%bash
set -e
mkdir -p /content/ckpt && cd /content/ckpt
if [ ! -f model.pth ]; then
  wget -q --show-progress -O cityscapes.zip \
    "https://github.com/czczup/ViT-Adapter/releases/download/v0.2.3/mask2former_beit_adapter_large_896_80k_cityscapes.zip"
  unzip -o cityscapes.zip
  find . -name '*.pth' -exec mv {} /content/ckpt/model.pth \;
  rm -f cityscapes.zip
fi
ls -lh /content/ckpt/model.pth


In [ ]:
!find /content/ckpt -type f -exec ls -lh {} \;

In [ ]:
!mv /content/ckpt/mask2former_beit_adapter_large_896_80k_cityscapes.pth.tar /content/ckpt/model.pth
!ls -lh /content/ckpt/model.pth

### 5. Write the video-inference script into the repo

In [ ]:
%%writefile /content/ViT-Adapter/segmentation/infer_video.py
"""
ViT-Adapter — Video Semantic Segmentation (frame-by-frame)
"""
import argparse
import os

# Force a headless matplotlib backend BEFORE mmseg imports pyplot.
os.environ["MPLBACKEND"] = "Agg"
import matplotlib  # noqa: E402
matplotlib.use("Agg")

import cv2  # noqa: E402
import mmcv  # noqa: E402
from mmcv import Config  # noqa: E402

import mmcv_custom   # noqa: F401,E402
import mmseg_custom  # noqa: F401,E402
from mmcv.runner import load_checkpoint  # noqa: E402
from mmseg.apis import inference_segmentor  # noqa: E402
from mmseg.models import build_segmentor  # noqa: E402
from mmseg.core.evaluation import get_classes, get_palette  # noqa: E402


def load_model(cfg, checkpoint_path, device, palette_name):
    cfg.model.pretrained = None
    cfg.model.train_cfg = None
    if cfg.model.get("backbone") is not None:
        cfg.model.backbone.pretrained = None
    model = build_segmentor(cfg.model, test_cfg=cfg.get("test_cfg"))
    ckpt = load_checkpoint(model, checkpoint_path, map_location="cpu")
    meta = ckpt.get("meta", {}) if isinstance(ckpt, dict) else {}
    model.CLASSES = meta.get("CLASSES", get_classes(palette_name))
    model.PALETTE = meta.get("PALETTE", get_palette(palette_name))
    model.cfg = cfg
    model.to(device)
    model.eval()
    return model


def parse_scale(s):
    w, h = s.lower().split("x")
    return (int(w), int(h))


def main():
    ap = argparse.ArgumentParser(description="ViT-Adapter video semantic segmentation")
    ap.add_argument("--config", required=True)
    ap.add_argument("--checkpoint", required=True)
    ap.add_argument("--video", required=True)
    ap.add_argument("--out", default="output_seg.mp4")
    ap.add_argument("--palette", default="cityscapes")
    ap.add_argument("--opacity", type=float, default=0.5)
    ap.add_argument("--scale", type=parse_scale, default=(1600, 896),
                    help="model input scale WxH; short side must be >= 896.")
    ap.add_argument("--stride", type=int, default=1,
                    help="run on every Nth frame (2 = half the frames, faster)")
    ap.add_argument("--seconds", type=float, default=0,
                    help="process only the first N seconds (0 = whole video)")
    ap.add_argument("--max-frames", type=int, default=0,
                    help="stop after this many processed frames (0 = all)")
    ap.add_argument("--device", default="cuda:0")
    args = ap.parse_args()

    cfg = Config.fromfile(args.config)
    for t in cfg.data.test.pipeline:
        if t.get("type") == "MultiScaleFlipAug":
            t["img_scale"] = args.scale

    print(f"Loading model on {args.device} (input scale {args.scale}) ...")
    model = load_model(cfg, args.checkpoint, args.device, args.palette)
    palette = get_palette(args.palette)

    reader = mmcv.VideoReader(args.video)
    src_fps = reader.fps or 25
    out_fps = src_fps / max(1, args.stride)
    src_limit = int(args.seconds * src_fps) if args.seconds else 0
    span = f", first {args.seconds:g}s ({src_limit} frames)" if src_limit else ""
    print(f"Video: {reader.width}x{reader.height} @ {src_fps:.2f}fps, "
          f"{len(reader)} frames{span} -> writing {args.out} @ {out_fps:.2f}fps")

    writer = None
    n = 0
    for idx, frame in enumerate(mmcv.track_iter_progress(reader)):
        if src_limit and idx >= src_limit:
            break
        if idx % args.stride != 0:
            continue
        result = inference_segmentor(model, frame)
        vis = model.show_result(frame, result, palette=palette,
                                opacity=args.opacity, show=False)
        if writer is None:
            h, w = vis.shape[:2]
            writer = cv2.VideoWriter(
                args.out, cv2.VideoWriter_fourcc(*"mp4v"), out_fps, (w, h))
        writer.write(vis)
        n += 1
        if args.max_frames and n >= args.max_frames:
            break

    if writer is not None:
        writer.release()
    print(f"\n[done] wrote {n} segmented frames -> {args.out}")


if __name__ == "__main__":
    main()

### 6. Upload your video  → saved as `/content/input.mp4`
Run this cell and pick **`sample_dashcam.mp4`** from your computer.
(Re-running is safe; if a file is already uploaded it's reused.)

In [ ]:
import os, shutil
from google.colab import files

DEST = '/content/input.mp4'
up = files.upload()          # opens the file picker
if up:
    src = list(up.keys())[0]
    if os.path.abspath(src) != DEST:
        shutil.move(src, DEST)
assert os.path.exists(DEST), "No video found — re-run and select your .mp4"
print('Ready:', DEST, '(%d KB)' % (os.path.getsize(DEST)//1024))

In [ ]:
!/usr/local/envs/vit/bin/pip install -q scipy

### 7. Run segmentation, then re-encode for in-browser playback
- `--scale` is the speed/quality knob: `1024x512` is the fast default that fits a T4;
  use `2048x1024` for full Cityscapes resolution (sharper, slower, more memory).
- `--stride 2` processes every other frame (~2× faster).
- The final `ffmpeg` step converts to **H.264 / yuv420p** so the result actually
  plays in Cell 8 and on any player.

In [ ]:
%%bash
set -e
source /usr/local/etc/profile.d/conda.sh
conda activate vit
export MPLBACKEND=Agg
cd /content/ViT-Adapter/segmentation

python infer_video.py \
  --config configs/cityscapes/mask2former_beit_adapter_large_896_80k_cityscapes_ss.py \
  --checkpoint /content/ckpt/model.pth \
  --video /content/input.mp4 \
  --out /content/output_raw.mp4 \
  --scale 1600x896 \
  --opacity 0.5 \
  --seconds 10 \
  --stride 3

ffmpeg -y -loglevel error -i /content/output_raw.mp4 \
  -c:v libx264 -pix_fmt yuv420p /content/output_seg.mp4
echo "Wrote /content/output_seg.mp4"
ls -lh /content/output_seg.mp4

### 8. Preview and download the segmented video

In [ ]:
from IPython.display import HTML
from base64 import b64encode
data = b64encode(open('/content/output_seg.mp4','rb').read()).decode()
HTML(f'<video width=720 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

In [ ]:
import os, glob
from base64 import b64encode
from IPython.display import HTML, display
from google.colab import files

ORIG = '/content/input.mp4'
SEG  = '/content/output_seg.mp4'
OUT  = '/content/comparison.mp4'
SECONDS = 10          # length of the comparison
FPS     = 10          # common playback fps for both sides (frames stay aligned)

assert os.path.exists(SEG), "Run Cell 5 (updated) + Cell 7 first to create output_seg.mp4"

# Pick a font that exists on Colab for the on-video labels (fallback: no labels).
cands = (glob.glob('/usr/share/fonts/**/DejaVuSans-Bold.ttf', recursive=True)
         or glob.glob('/usr/share/fonts/**/*Bold.ttf', recursive=True)
         or glob.glob('/usr/share/fonts/**/*.ttf', recursive=True))
font = cands[0] if cands else None

def label(txt):
    if not font:
        return ''
    return (f",drawtext=fontfile='{font}':text='{txt}':x=(w-tw)/2:y=18:"
            f"fontsize=34:fontcolor=white:box=1:boxcolor=black@0.5:boxborderw=12")

# Left  = original trimmed to first 10s; Right = segmented result.
# Both normalized to height 720 @ same fps so the two halves line up, then hstacked.
filt = (f"[0:v]trim=0:{SECONDS},setpts=PTS-STARTPTS,fps={FPS},scale=-2:720{label('Original')}[l];"
        f"[1:v]fps={FPS},scale=-2:720{label('ViT-Adapter (Cityscapes)')}[r];"
        f"[l][r]hstack=inputs=2[v]")

!ffmpeg -y -loglevel error -i {ORIG} -i {SEG} -filter_complex "{filt}" -map "[v]" -c:v libx264 -pix_fmt yuv420p {OUT}
print('Wrote', OUT, '(%.1f MB)' % (os.path.getsize(OUT) / 1e6))

# Preview inline, then download.
data = b64encode(open(OUT, 'rb').read()).decode()
display(HTML(f'<video width=960 controls><source src="data:video/mp4;base64,{data}" type="video/mp4"></video>'))
files.download(OUT)

In [ ]:
from google.colab import files
files.download('/content/output_seg.mp4')

---
## 🛠️ Troubleshooting

| Symptom | Fix |
|---|---|
| `torch.cuda.is_available() == False` / `nvidia-smi` empty | You're on a CPU runtime. `Runtime ▸ Change runtime type ▸ GPU`, then **Runtime ▸ Restart** and run from Cell 0. |
| Cell 1 "session crashed / restarting" | That's normal — `condacolab` restarts the runtime. Continue from Cell 2. |
| `unsupported GNU version` while building the op (Cell 3) | Already handled (we force gcc-10). If it still appears, re-run Cell 3. |
| `MultiScaleDeformableAttention` import / `undefined symbol` | Runtime was recycled. Re-run Cells 2 → 3 (env + op build) in order. |
| **CUDA out of memory** in Cell 7 | Lower `--scale` (e.g. `768x384`) and/or add `--stride 2`. |
| Cell 8 video is black / won't play | The H.264 re-encode in Cell 7 fixes this; make sure Cell 7 finished without error. |
| Want a quick test first | Add `--max-frames 60` in Cell 7 to process only the first ~60 frames. |

**Different scene type?** This is the Cityscapes (driving) model — ideal for your
dashcam clip. For general scenes, swap to the ADE20K config + checkpoint in Cells 4 & 7.
